In [1]:
import pandas as pd

In [2]:
df_irve = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435'
)


/tmp/ipykernel_41605/4144409297.py:1: DtypeWarning: Columns (12,18,19,20,21,22,24,25,29,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df_irve = pd.read_csv(


In [3]:
print(df_irve['code_insee_commune'].nunique())

10973


In [4]:
from shapely.geometry import Point
import geopandas as gpd

df_irve["geometry"] = df_irve.apply(
    lambda row: Point(row["consolidated_longitude"], row["consolidated_latitude"]),
    axis=1
)

gdf_irve = gpd.GeoDataFrame(df_irve, geometry="geometry", crs="EPSG:4326")


from cartiflette import carti_download

# Liste des départements (France entière)
departements = [str(i).zfill(2) for i in range(1, 96)]  # 01 à 95

# Téléchargement des communes françaises
communes = carti_download(
    crs=4326,                     # coordonnées GPS (longitude/latitude)
    borders="COMMUNE",             # niveau géographique : commune
    filter_by="DEPARTEMENT",
    values=departements,
    vectorfile_format="geojson",   # format pratique pour GeoPandas
    source="EXPRESS-COG-CARTO-TERRITOIRE",
    year=2022
)

communes = communes.to_crs(gdf_irve.crs)

gdf_result = gpd.sjoin(gdf_irve, communes[['INSEE_COM', 'geometry']], how="left", predicate="within")

df_irve["code_geo_manquant"] = df_irve["code_insee_commune"].fillna(gdf_result["INSEE_COM"])
df_irve["code_geo_total"] = gdf_result["INSEE_COM"]

There was an error while reading the file from the URL: https://minio.lab.sspcloud.fr/projet-cartiflette/production/provider=IGN/dataset_family=ADMINEXPRESS/source=EXPRESS-COG-CARTO-TERRITOIRE/year=2022/administrative_level=COMMUNE/crs=4326/DEPARTEMENT=20/vectorfile_format=geojson/territory=metropole/simplification=0/raw.geojson
Error message: '/vsimem/pyogrio_9a041c0fd8314d378eda37d857408abc' not recognized as being in a supported file format.; It might help to specify the correct driver explicitly by prefixing the file path with '<DRIVER>:', e.g. 'CSV:path'.


In [5]:
df_ve = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/90e0d717-deda-4bdc-9987-f82faac5bc93',
 encoding='latin-1'
)

/tmp/ipykernel_41605/2410490780.py:1: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ve = pd.read_csv(


In [6]:
df_revenus = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/516130bc-4dcb-47f5-8347-ae96553c43ab',sep=';'
)

/tmp/ipykernel_41605/882139779.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_revenus = pd.read_csv(


In [7]:
# Combien de codes a-t-on réussi à récupérer ?
print(f"Codes manquants avant : {df_irve['code_insee_commune'].isna().sum()}")
print(f"Codes manquants après : {df_irve['code_geo_manquant'].isna().sum()}")
print(f"Codes manquants après : {df_irve['code_geo_total'].isna().sum()}")

print(df_irve['code_geo_manquant'].nunique())
print(df_irve['code_geo_total'].nunique())
print(df_irve['code_insee_commune'].nunique())

Codes manquants avant : 64848
Codes manquants après : 1233
Codes manquants après : 1761
14386
10253
10973


In [12]:
len(set(df_irve["code_insee_commune"].dropna()))

10973

In [10]:
len(set(df_irve["code_geo_manquant"].dropna()))

14386

In [13]:
len(set(df_irve["code_geo_total"].dropna()))

10253

In [15]:
len(set(df_ve["CODGEO"].dropna()))

35206

In [16]:
len(set(df_revenus["Code géographique"].dropna()))

34926

In [17]:
df_ve["CODGEO"] = df_ve["CODGEO"].dropna().astype(str).str.zfill(5)
df_revenus["Code géographique"] = df_revenus["Code géographique"].dropna().astype(str).str.zfill(5)
df_irve["code_insee_commune"] = df_irve["code_insee_commune"].dropna().astype(str).str.zfill(5)
df_irve["code_geo_manquant"] = df_irve["code_geo_manquant"].dropna().astype(str).str.zfill(5)
df_irve["code_geo_total"] = df_irve["code_geo_total"].dropna().astype(str).str.zfill(5)

In [ ]:
len(set(df_ve["CODGEO"]) - set(df_revenus["Code géographique"]))

275

In [23]:
len(set(df_ve["CODGEO"]) - set(df_irve["code_geo_manquant"]))

26004

ça me parait beaucoup

In [25]:
len(set(df_ve["CODGEO"]) - set(df_irve["code_geo_total"]))

24944

c'est mieux

In [27]:
len(set(df_revenus["Code géographique"]) - set(df_irve["code_geo_manquant"]))

25740

beaucoup

In [28]:
len(set(df_revenus["Code géographique"]) - set(df_irve["code_geo_total"]))

24676

mieux

In [29]:
len(set(df_revenus["Code géographique"]) - set(df_ve["CODGEO"]))

4

In [30]:
len(set(df_irve["code_geo_manquant"]) - set(df_revenus["Code géographique"]))

5201

beaucoup

In [31]:
len(set(df_irve["code_geo_total"]) - set(df_revenus["Code géographique"]))

4

mieux

In [24]:
len(set(df_irve["code_geo_manquant"]) - set(df_ve["CODGEO"]))

5194

ça me parait beaucoup

In [26]:
len(set(df_irve["code_geo_total"]) - set(df_ve["CODGEO"]))

1

Les correspondances sont bien mieux quand on recalcule tous les codes géo, ceux initiaux doivent être différents de mes autres bases de données.

c'est peut-être des histoires d'arrondissements, ou code géo pas de la même année...